# Análisis científico de resultados F0 (protocolo v11)

Este notebook documenta el proceso completo de análisis de los experimentos F0 bajo el protocolo v11, desde la carga y exploración de los datos hasta la generación automática del informe de hallazgos científicos.

**Estructura:**
1. Importar librerías necesarias
2. Cargar y preparar los datos
3. Exploración y visualización de los datos
4. Preprocesamiento de los datos
5. Entrenamiento de modelos (si aplica)
6. Evaluación de resultados
7. Generación automática del documento de análisis de hallazgos


## 1. Importar librerías necesarias

Importamos las librerías requeridas para análisis de datos, visualización y generación de informes científicos.

In [ ]:
# Importar librerías para análisis y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path
# Para generación de informes
from matplotlib.backends.backend_pdf import PdfPages


## 2. Cargar y preparar los datos

Cargamos los archivos JSON y CSV generados por los experimentos F0 (protocolo v11) para grid8 y grid16, y mostramos una vista preliminar de los datos.

In [ ]:
# Definir rutas a los archivos de resultados
raw_dir = Path('../raw')

# Archivos para grid8 y grid16
files = {
    'grid8': {
        'json': raw_dir / 'grid8_risklow_seed42_v11.json',
        'csv': raw_dir / 'grid8_risklow_seed42_v11_episodes.csv'
    },
    'grid16': {
        'json': raw_dir / 'grid16_risklow_seed42_v11.json',
        'csv': raw_dir / 'grid16_risklow_seed42_v11_episodes.csv'
    }
}

# Cargar CSVs en DataFrames
df_grid8 = pd.read_csv(files['grid8']['csv'])
df_grid16 = pd.read_csv(files['grid16']['csv'])

# Mostrar primeras filas de cada DataFrame
display(df_grid8.head())
display(df_grid16.head())

## 3. Exploración y visualización de los datos

Análisis exploratorio de las métricas principales por agente y grid. Se incluyen histogramas, diagramas de caja y gráficos comparativos para reward, flexibilidad, robustez, risk_effective, surprise y PGF.

In [ ]:
# Función para graficar métricas por agente y grid
def plot_metric(df, metric, grid_label):
    plt.figure(figsize=(8,5))
    sns.boxplot(x='Agente', y=metric, data=df)
    plt.title(f'{metric} por agente ({grid_label})')
    plt.ylabel(metric)
    plt.xlabel('Agente')
    plt.show()

# Métricas principales a analizar
metrics = ['Recompensa', 'Flexibilidad', 'Robustez', 'RiskEffective_Avg', 'Surprise_Avg', 'PGF_Bruto_Avg', 'PGF_Costo_Avg']

for metric in metrics:
    plot_metric(df_grid8, metric, 'grid8')
    plot_metric(df_grid16, metric, 'grid16')

## 4. Preprocesamiento de los datos

Limpieza de datos, verificación de valores nulos y preparación para análisis estadístico y modelado.

In [ ]:
# Verificar valores nulos y tipos de datos
def check_and_clean(df, label):
    print(f'--- {label} ---')
    print(df.info())
    print(df.isnull().sum())
    # Si hay nulos, imputar o eliminar según el caso
    df_clean = df.dropna()
    return df_clean

df_grid8_clean = check_and_clean(df_grid8, 'grid8')
df_grid16_clean = check_and_clean(df_grid16, 'grid16')

## 5. Entrenamiento de modelos (opcional)

Si se requiere, aquí se pueden entrenar modelos de regresión o clasificación para explorar relaciones entre métricas o predecir resultados a partir de las variables instrumentadas.

In [ ]:
# Ejemplo: regresión lineal para explorar relación entre risk_effective y recompensa
from sklearn.linear_model import LinearRegression

for df, label in zip([df_grid8_clean, df_grid16_clean], ['grid8', 'grid16']):
    X = df[['RiskEffective_Avg']]
    y = df['Recompensa']
    model = LinearRegression().fit(X, y)
    print(f'[{label}] Coeficiente: {model.coef_[0]:.3f}, Intercepto: {model.intercept_:.3f}, Score R2: {model.score(X, y):.3f}')

## 6. Evaluación de resultados

Evaluamos el desempeño de los agentes y condiciones experimentales usando métricas descriptivas y visualizaciones comparativas. Se resumen los hallazgos clave en tablas y gráficos.

In [ ]:
# Resumen estadístico por agente y grid
def resumen_agente(df, label):
    print(f'--- {label} ---')
    print(df.groupby('Agente').agg({
        'Recompensa': ['mean', 'std'],
        'Flexibilidad': 'mean',
        'Robustez': 'mean',
        'RiskEffective_Avg': 'mean',
        'Surprise_Avg': 'mean',
        'PGF_Bruto_Avg': 'mean',
        'PGF_Costo_Avg': 'mean'
    }))

resumen_agente(df_grid8_clean, 'grid8')
resumen_agente(df_grid16_clean, 'grid16')

## 7. Generación automática del documento de análisis de hallazgos

En esta sección se genera un informe PDF con los resultados, visualizaciones y hallazgos principales del experimento F0 (protocolo v11).

In [ ]:
# Ejemplo de generación de informe PDF con matplotlib
with PdfPages('informe_F0_v11.pdf') as pdf:
    for metric in metrics:
        plt.figure(figsize=(8,5))
        sns.boxplot(x='Agente', y=metric, data=df_grid8_clean)
        plt.title(f'{metric} por agente (grid8)')
        pdf.savefig()
        plt.close()
        plt.figure(figsize=(8,5))
        sns.boxplot(x='Agente', y=metric, data=df_grid16_clean)
        plt.title(f'{metric} por agente (grid16)')
        pdf.savefig()
        plt.close()
    # Página resumen
    plt.figure(figsize=(10,2))
    plt.text(0.1, 0.5, 'Informe automático F0 v11\nResultados y hallazgos principales generados desde el notebook.', fontsize=14)
    plt.axis('off')
    pdf.savefig()
    plt.close()
print('Informe PDF generado: informe_F0_v11.pdf')

## 8. Exportaci?n de figuras y tablas

Esta secci?n exporta tablas resumen de m?tricas por agente y grid a CSV, complementando el informe PDF generado anteriormente.


In [ ]:
# Exportar tabla resumen de m?tricas por agente y grid a CSV
summary_rows = []
for df, label in [(df_grid8_clean, 'grid8'), (df_grid16_clean, 'grid16')]:
    agg = df.groupby('Agente').agg({
        'Recompensa': ['mean', 'std'],
        'Flexibilidad': 'mean',
        'Robustez': 'mean',
        'RiskEffective_Avg': 'mean',
        'Surprise_Avg': 'mean',
        'PGF_Bruto_Avg': 'mean',
        'PGF_Costo_Avg': 'mean',
    })
    # Aplanar columnas multi-nivel y a?adir etiqueta de grid
    agg.columns = ['_'.join([str(c) for c in cols if c]) for cols in agg.columns.values]
    agg = agg.reset_index()
    agg.insert(0, 'Grid', label)
    summary_rows.append(agg)

summary_df = pd.concat(summary_rows, ignore_index=True)
summary_path = Path('resumen_F0_v11_metricas.csv')
summary_df.to_csv(summary_path, index=False)
print('Tabla resumen guardada en:', summary_path)
summary_df
